<a href="https://colab.research.google.com/github/MElsdk-lab/Biochar_forest_estimation/blob/main/Notebook_4_Forest_type_composition_sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# NOTEBOOK 4 — GEE Forest compposition and Type Computation - Hansen 7 GLC
# University of Pittsburgh | Biochar Feedstock Methodology
# ============================================================

In [1]:
# ── CELL 1: Install Libraries ─────────────────────────────────────────────────
!pip install -q earthengine-api geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.7 MB/s eta 0:00:00


In [2]:
# ── CELL 2: conect to drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_FOLDER = '/content/drive/MyDrive/BiocharProject/'
print('✅ connected to drive')

Mounted at /content/drive
✅ connected to drive


In [3]:
# ── CELL 3: clone repo if not already cloned ─────────────────────
import os
import getpass
import subprocess


if not os.path.exists('/content/Biochar_forest_estimation'):
    PAT = getpass.getpass('Enter PAT: ')
    !git config --global user.email "sdkmajd@gmail.com"
    !git config --global user.name "MElsdk-lab"
    subprocess.run(
        f'git clone https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git',
        shell=True
    )

%cd /content/Biochar_forest_estimation/
!git fetch origin
!git reset --hard origin/main
print('✅ Ready')

Enter PAT: ··········
/content/Biochar_forest_estimation
HEAD is now at 9511a8b only for Oregon state
✅ Ready


In [4]:
# ── CELL 4: import libraries and data from data_config ─────────────────────

import ee
import geemap
import pandas as pd
import time
from data_config import FAO_LSIB_REGION, us_state_names, get_all_countries, build_country_lookup, forest_bins

print('✅ Libraries imported')
print('✅ Data config loaded')

✅ Libraries imported
✅ Data config loaded


In [5]:
# ── CELL 5: Authenticate and initialize Earth Engine & Load Datasets  ─────────────────────
try:
    ee.Initialize(project='spry-blade-487218-n0')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='spry-blade-487218-n0')  # ← add project here too

Hansen_GFC_2024      = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")
GLC_FSC30D_annual    = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/annual')
GLC_FSC30D_five_year = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/five-years-map')

print('✅ Hansen GFC 2024 loaded')
print('✅ GLC FCS30D annual loaded')
print('✅ GLC FCS30D five-year loaded')


✅ Hansen GFC 2024 loaded
✅ GLC FCS30D annual loaded
✅ GLC FCS30D five-year loaded


In [6]:
# ── CELL 6: processing datasets ───────────────────────────────────────

# Select & Mask Hansen Bands
treecover2000 = Hansen_GFC_2024.select('treecover2000')
datamask      = Hansen_GFC_2024.select('datamask')

treecover2000_masked = treecover2000.updateMask(treecover2000.gt(0)) \
                                    .updateMask(datamask.eq(1))

print('✅ treecover2000 masked')

#Load GLC FCS30D Year 2000

glc_2000        = GLC_FSC30D_annual.mosaic().select('b1')
glc_2000_forest = glc_2000.gte(51).And(glc_2000.lte(92)).selfMask()

print('✅ GLC FCS30D 2000 loaded')
print('✅ GLC forest mask created (classes 51-92)')

# Forest Class Definitions
forestClasses = [
    {'code': 51, 'color': '4c7300', 'name': '51 Open evergreen broadleaved'},
    {'code': 52, 'color': '006400', 'name': '52 Closed evergreen broadleaved'},
    {'code': 61, 'color': 'a8c800', 'name': '61 Open deciduous broadleaved'},
    {'code': 62, 'color': '00a000', 'name': '62 Closed deciduous broadleaved'},
    {'code': 71, 'color': '005000', 'name': '71 Open evergreen needleleaved'},
    {'code': 72, 'color': '003c00', 'name': '72 Closed evergreen needleleaved'},
    {'code': 81, 'color': '286400', 'name': '81 Open deciduous needleleaved'},
    {'code': 82, 'color': '285000', 'name': '82 Closed deciduous needleleaved'},
    {'code': 91, 'color': 'a0b432', 'name': '91 Open mixed forest'},
    {'code': 92, 'color': '788200', 'name': '92 Closed mixed forest'},
]

print(f'✅ {len(forestClasses)} forest classes defined')

✅ treecover2000 masked
✅ GLC FCS30D 2000 loaded
✅ GLC forest mask created (classes 51-92)
✅ 10 forest classes defined


In [42]:
# ── CELL 7: GEE Functions — Countries Forest Cover Area by Type ───────────────

def get_forest_cover_area_type_country(country_feature, forest_classification):
    """
    For one country feature, compute forest area (ha) per GLC forest type.
    Returns a GEE FeatureCollection — one Feature with one column per class.
    """
    class_images = []
    for fc in forest_classification:
        class_mask = glc_2000.eq(fc['code']).selfMask()
        class_area = class_mask.multiply(ee.Image.pixelArea().divide(1e10)).rename(fc['name'])
        class_images.append(class_area)

    multi_band_image = ee.Image.cat(class_images)

    region_area = multi_band_image.reduceRegions(
        collection=ee.FeatureCollection([country_feature]),
        reducer=ee.Reducer.sum(),
        scale=30
    )
    return region_area


def export_forest_cover_area_type_all_countries(selected_regions, forest_classification):
    """
    Loop over all countries, build one combined FeatureCollection,
    and submit a single export task to Drive.
    """
    all_countries = get_all_countries(selected_regions)
    lsib_fao = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017') \
                  .filter(ee.Filter.inList('country_na', all_countries))
    country_list = lsib_fao.toList(lsib_fao.size())
    n = lsib_fao.size().getInfo()

    combined = ee.FeatureCollection([])
    for i in range(n):
        country_feature = ee.Feature(country_list.get(i))
        fc = get_forest_cover_area_type_country(country_feature, forest_classification)
        combined = combined.merge(fc)

    selectors = ['country_na'] + [fc['name'] for fc in forest_classification]

    country_task = ee.batch.Export.table.toDrive(
        collection=combined,
        description='forest_cover_area_type_all_countries',
        folder='GEE_exports',
        fileNamePrefix='forest_cover_area_type_all_countries',
        fileFormat='CSV',
        selectors=selectors
    )
    country_task.start()
    print('✅ Single export task submitted: forest_cover_area_type_all_countries')
    return country_task


print('✅ get_forest_cover_area_type_country() defined')
print('✅ export_forest_cover_area_type_all_countries() defined')

✅ get_forest_cover_area_type_country() defined
✅ export_forest_cover_area_type_all_countries() defined


In [43]:
# ── CELL 8: GEE Functions — States Forest Cover Area by Type ──────────────────

def get_forest_cover_area_type_state(state_feature, forest_classification):
    """
    For one state feature, compute forest area (ha) per GLC forest type.
    Returns a GEE FeatureCollection — one Feature with one column per class.
    """
    class_images = []
    for fc in forest_classification:
        class_mask = glc_2000.eq(fc['code']).selfMask()
        class_area = class_mask.multiply(ee.Image.pixelArea().divide(1e10)).rename(fc['name'])
        class_images.append(class_area)

    multi_band_image = ee.Image.cat(class_images)

    region_area = multi_band_image.reduceRegions(
        collection=ee.FeatureCollection([state_feature]),
        reducer=ee.Reducer.sum(),
        scale=30
    )
    return region_area


def export_forest_cover_area_type_all_states(selected_states, forest_classification):
    """
    Loop over all states, build one combined FeatureCollection,
    and submit a single export task to Drive.
    """
    tiger_states = ee.FeatureCollection('TIGER/2018/States') \
                     .filter(ee.Filter.inList('NAME', selected_states))
    state_list = tiger_states.toList(tiger_states.size())
    n = tiger_states.size().getInfo()

    combined = ee.FeatureCollection([])
    for i in range(n):
        state_feature = ee.Feature(state_list.get(i))
        fc = get_forest_cover_area_type_state(state_feature, forest_classification)
        combined = combined.merge(fc)

    selectors = ['NAME'] + [fc['name'] for fc in forest_classification]

    state_task = ee.batch.Export.table.toDrive(
        collection=combined,
        description='forest_cover_area_type_all_states',
        folder='GEE_exports',
        fileNamePrefix='forest_cover_area_type_all_states',
        fileFormat='CSV',
        selectors=selectors
    )
    state_task.start()
    print('✅ Single export task submitted: forest_cover_area_type_all_states')
    return state_task


print('✅ get_forest_cover_area_type_state() defined')
print('✅ export_forest_cover_area_type_all_states() defined')

✅ get_forest_cover_area_type_state() defined
✅ export_forest_cover_area_type_all_states() defined


In [ ]:
# ── CELL 9: Run Exports ────────────────────────────────────────────────────────

print('── Submitting country forest cover bins task ──')
country_bins_task = export_forest_cover_bins_all_countries(FAO_LSIB_REGION, forest_bins)

print('\n── Submitting state forest cover bins task ──')
state_bins_task = export_forest_cover_bins_all_states(us_state_names, forest_bins)


In [ ]:
# ── CELL 10: Monitor Tasks ─────────────────────────────────────────────────────
import time

all_tasks = [country_bins_task, state_bins_task]

for i in range(30):
    print(f'\n── Check {i+1} ──')
    all_done = True
    for task in all_tasks:
        status = task.status()
        print(f"  {status['description']}: {status['state']}")
        if status['state'] not in ['COMPLETED', 'FAILED']:
            all_done = False
    if all_done:
        print('\n✅ All tasks completed!')
        break
    time.sleep(60)


In [ ]:
# ── CELL 11: Copy GEE Exports from Drive to GitHub Repo ───────────────────────
import shutil

files = [
    'forest_cover_bins_all_countries.csv',
    'forest_cover_bins_all_states.csv',
]

for f in files:
    shutil.copy(GEE_FOLDER + f, DATA_FOLDER + f)
    print(f'✅ Copied {f}')

In [ ]:
# ── CELL 12: Push results to GitHub ────────────────────────────────
import subprocess

# ask for PAT only if not already defined
if 'PAT' not in globals():
    import getpass
    PAT = getpass.getpass('Enter PAT: ')

%cd /content/Biochar_forest_estimation/
!git add .
!git commit -m "update results"

result = subprocess.run(
    f'git push https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git main',
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print('✅ Pushed to GitHub')
else:
    print('❌ Push failed')
    print(result.stderr)